### Links and Resources
| Concept | Description |
|---|---|
| [Python UDFs in Unity Catalog](https://docs.databricks.com/en/udf/unity-catalog-python-udfs.html) | Register Python as a governed function |
| [`CREATE FUNCTION ... LANGUAGE PYTHON`](https://docs.databricks.com/en/sql/language-manual/sql-ref-syntax-ddl-create-sql-function.html#parameters) | DDL syntax |
| [Limits & supported runtimes](https://docs.databricks.com/en/udf/unity-catalog-python-udfs.html#limitations) | What Python you can use |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# Python UDFs in Unity Catalog

Notebook 11 created Python UDFs that live only in the current SparkSession. UC Python UDFs solve the same problem **persistently and securely**: register once, govern via `GRANT EXECUTE`, call from any notebook, job, SQL warehouse, or BI tool.

The DDL is the same `CREATE FUNCTION` you saw in notebook 13, but with `LANGUAGE PYTHON` and a `def` body.

In [ ]:
CATALOG = "main"
SCHEMA  = "default"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA  {SCHEMA}")

## Scalar Python UDF in UC

In [ ]:
spark.sql("""
CREATE OR REPLACE FUNCTION email_domain(email STRING)
RETURNS STRING
LANGUAGE PYTHON
AS $$
    if email is None:
        return None
    return email.lower().split("@")[-1]
$$
""")

spark.sql("SELECT email_domain('Alice@Example.COM') AS d").show()

## Use in a query — like any other function

In [ ]:
people = spark.createDataFrame([
    (1, "Alice", "alice@example.com"),
    (2, "Bob",   "BOB@EXAMPLE.ORG"),
    (3, "Carol", None),
], ["id", "name", "email"])
people.createOrReplaceTempView("people")

spark.sql("SELECT id, name, email_domain(email) AS domain FROM people").show()

## Multiple parameters and structured returns

In [ ]:
spark.sql("""
CREATE OR REPLACE FUNCTION classify_score(score INT, threshold INT)
RETURNS STRUCT<bucket: STRING, distance: INT>
LANGUAGE PYTHON
AS $$
    if score is None:
        return {"bucket": "unknown", "distance": None}
    if score >= threshold:
        return {"bucket": "above", "distance": score - threshold}
    return {"bucket": "below", "distance": threshold - score}
$$
""")

spark.sql("SELECT classify_score(85, 70) AS r").show(truncate=False)

## What's in the Python environment

UC Python UDFs run in a **sandboxed environment** with a curated standard library. Notable points:

- The Python standard library is mostly available (`re`, `math`, `json`, `datetime`, `hashlib`, ...).
- **Network calls are blocked** by default — no `requests`, no API calls.
- **Filesystem access is blocked** — no reading workspace files / volumes from the UDF body.
- **Custom imports**: classic UC Python UDFs are stdlib-only. For sklearn/numpy/etc., use **service credentials and dependencies via Python UDFs on serverless** or fall back to a session-local `@F.udf` (notebook 11) or a Pandas UDF (notebook 12).

Always check the current Databricks docs — the supported package list expands over time.

In [ ]:
spark.sql("""
CREATE OR REPLACE FUNCTION sha256_hex(s STRING)
RETURNS STRING
LANGUAGE PYTHON
AS $$
    import hashlib
    if s is None:
        return None
    return hashlib.sha256(s.encode("utf-8")).hexdigest()
$$
""")

spark.sql("SELECT sha256_hex('hello') AS h").show(truncate=False)

## Permissions — same model as SQL UDFs

```sql
GRANT EXECUTE ON FUNCTION main.default.email_domain TO `analysts`;
```

The principal also needs `USAGE` on the schema and `USE CATALOG` on the catalog. **Owners can read the body** by default — be careful with secrets in the source.

## When to choose UC Python UDF vs. classic UDF vs. Pandas UDF

| Need | Pick |
|---|---|
| One-line transformation, used across many notebooks/jobs | **UC Python UDF** |
| Heavy numpy / sklearn / model inference | **Pandas UDF** (notebook 12) |
| Throwaway logic, one notebook | **Classic `@F.udf`** (notebook 11) |
| Pure SQL transformation, governable | **UC SQL UDF** (notebook 13) |
| ML model server / external API call | UDF can't do this — use a Spark Connect job or `foreachBatch` |

## Limits to know

- Single-row scalar invocation — **not vectorized** like Pandas UDFs.
- Sandboxed Python — limited package set, no network, no FS.
- Body must be valid Python (the dollar-quoted block is parsed at registration).
- Versioned: `CREATE OR REPLACE` increments the version. Older queries pinned via `[VERSION ...]` for reproducibility (Databricks-specific feature).

## Cleanup

In [ ]:
for fn in ["email_domain", "classify_score", "sha256_hex"]:
    spark.sql(f"DROP FUNCTION IF EXISTS {CATALOG}.{SCHEMA}.{fn}")
print("dropped")

## Takeaways

- UC Python UDFs = `CREATE FUNCTION ... LANGUAGE PYTHON AS $$ ... $$`.
- Persisted, governed (`GRANT EXECUTE`), reusable across notebooks, jobs, SQL warehouses, BI tools.
- Sandboxed: no network, no filesystem, curated stdlib only.
- Scalar/per-row — for vectorized work use Pandas UDFs (notebook 12).
- Same NULL caveat as classic Python UDFs — guard explicitly.